<a href="https://colab.research.google.com/github/nored/multimodal-disaster-classification/blob/main/determinism_retest_500.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Determinism Retest: 500-Sample Reproducibility Verification

**Purpose:** Re-classify 500 randomly sampled items from the original run and compare predictions to verify deterministic behavior at `temperature=0`.

**Requirements:**
- Google Colab with **A100 GPU** runtime
- The `all_results.csv` file from the original classification run

**Runtime:** ~30-40 minutes total (20 min setup + 10-15 min retest)

**Output:** `determinism_retest_500.json` report + paper-ready sentence for Section 3.4

## Cell 1: Setup — Build llama.cpp, Download Model & Dataset
This takes ~15-20 minutes. Go get coffee.

In [1]:
# =============================================================
# CELL 1: SETUP — Build llama.cpp, download model and dataset
# =============================================================

import os

# Build llama.cpp with CUDA
# Find the commit from May 22, 2025 and rebuild
%cd /content/llama.cpp

# Find the latest commit on or before May 22
!git log --before="2025-05-23" --oneline -5

# Checkout that commit
import subprocess
result = subprocess.run(['git', 'log', '--before=2025-05-23', '--format=%H', '-1'],
                       capture_output=True, text=True, cwd='/content/llama.cpp')
commit = result.stdout.strip()
print(f"Checking out: {commit}")
!git checkout {commit}

# Rebuild
!rm -rf build
!cmake -B build -DGGML_CUDA=ON
!cmake --build build --config Release -- -j$(nproc)

# Download model
!pip install -q huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1
!huggingface-cli download bartowski/mistralai_Mistral-Small-3.1-24B-Instruct-2503-GGUF \
    --include "mistralai_Mistral-Small-3.1-24B-Instruct-2503-Q6_K_L.gguf" \
    --local-dir . --local-dir-use-symlinks False
!huggingface-cli download bartowski/mistralai_Mistral-Small-3.1-24B-Instruct-2503-GGUF \
    --include "mmproj-mistralai_Mistral-Small-3.1-24B-Instruct-2503-f16.gguf" \
    --local-dir . --local-dir-use-symlinks False

# Download CrisisMMD dataset
!wget -q https://antidote.cloud/f/d5d4d7c6e17d41198522/?dl=1 -O CrisisMMD_v2.0.tar.gz
!tar -xf CrisisMMD_v2.0.tar.gz

# Create results directory
os.makedirs('/content/llama.cpp/results', exist_ok=True)

print('\n Setup complete.')

Cloning into '/content/llama.cpp'...
remote: Enumerating objects: 77734, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 77734 (delta 65), reused 43 (delta 43), pack-reused 77636 (from 4)
Receiving objects: 100% (77734/77734), 285.83 MiB | 43.29 MiB/s, done.
Resolving deltas: 100% (56228/56228), done.
/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identific

## Cell 2: Start Inference Server
Wait ~20 seconds for the server to load the model into GPU memory.

In [20]:
# =============================================================
# CELL 2: START INFERENCE SERVER
# =============================================================

import subprocess, time, requests

# Kill any existing server
!pkill -f llama-server 2>/dev/null
!sleep 2

# Start server with identical parameters to original run
!nohup /content/llama.cpp/build/bin/llama-server \
    --chat-template mistral-v7-tekken \
    -m /content/llama.cpp/mistralai_Mistral-Small-3.1-24B-Instruct-2503-Q6_K_L.gguf \
    --mmproj /content/llama.cpp/mmproj-mistralai_Mistral-Small-3.1-24B-Instruct-2503-f16.gguf \
    --host 0.0.0.0 \
    --port 8000 \
    -ngl 99 \
    -c 24576 \
    -b 1024 \
    -ub 512 \
    -cb \
    -t 8 \
    --mlock \
    --parallel 16 \
    > /content/llama.cpp/server.log 2>&1 &

# Wait for server to be ready
print('Waiting for server to start...')
for i in range(60):
    time.sleep(2)
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'\n Server ready after {(i+1)*2}s')
            break
    except:
        print('.', end='', flush=True)
else:
    print('\n Server failed to start. Check server.log')
    !tail -20 /content/llama.cpp/server.log

^C
Waiting for server to start...

 Server ready after 8s


## Cell 3: Upload Original Results & Run Retest
Upload your `all_results.csv` when prompted, then the retest runs automatically.

In [21]:
# =============================================================
# CELL 3: UPLOAD ORIGINAL RESULTS & RUN 500-SAMPLE RETEST
# =============================================================

import pandas as pd
import requests
import base64
import json
import os
import io
import re
import time
import random
import traceback
from PIL import Image
from tqdm.notebook import tqdm
from google.colab import files

# --- Upload original results ---
print('Upload your all_results.csv from the original run:')
uploaded = files.upload()
upload_filename = list(uploaded.keys())[0]
RESULTS_FILE = f'/content/{upload_filename}'
with open(RESULTS_FILE, 'wb') as f:
    f.write(uploaded[upload_filename])
print(f'Uploaded: {upload_filename}')

# --- Classification function (identical to original) ---
def classify_with_schema(tweet_text, image_path, server_endpoint="http://localhost:8000/chat/completions"):
    """
    Classify using the OpenAI-compatible /chat/completions endpoint with a JSON schema.
    This is the IDENTICAL function from the original classification notebook.
    """
    try:
        image = Image.open(image_path)
        if image.mode == 'RGBA':
            background = Image.new('RGB', image.size, (255, 255, 255))
            background.paste(image, mask=image.split()[3])
            image = background
        elif image.mode != 'RGB':
            image = image.convert('RGB')
        image = image.resize((512, 512))
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=85)
        image_base64 = base64.b64encode(buffer.getvalue()).decode('utf-8')

        request_data = {
            "model": "mistral-v7-tekken",
            "messages": [
                {
                    "role": "system",
                    "content": """You are an expert humanitarian disaster analyst with extensive experience in classifying disaster-related content. Your task is to accurately classify tweets and images based on objective evidence rather than emotional responses.

When analyzing images:
1. First, carefully examine the entire image for ALL visual evidence of disaster impacts
2. Look for people, damage, rescue activities, and other humanitarian elements BEFORE considering the "not_humanitarian" category
3. Pay particular attention to subtle signs of affected individuals, even if they are not prominently displayed
4. Only classify as "not_humanitarian" when you are certain there is no disaster-related content

Remember: Images showing ANY signs of disaster impact, even if subtle or in the background, should be classified as "informative" and given an appropriate category other than "not_humanitarian"."""
                },
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": f"""Tweet: "{tweet_text}"

CLASSIFICATION CRITERIA:

STEP 1: Analyze the TWEET TEXT first:
--------------------------------------
1. Tweet informativeness - determine if the tweet contains SPECIFIC information:
   - informative: Contains SPECIFIC disaster impact/response evidence, facts, or details
   - not_informative: Generic statements, emotions only, unrelated content, no specific details

2. Tweet category - identify the DOMINANT content (choose only ONE):
   - affected_individuals: Mentions displaced people, survivors, emotional responses (NOT injured/dead)
   - infrastructure_and_utility_damage: References damaged buildings, roads, bridges, utilities
   - injured_or_dead_people: Reports injuries, deaths, or specific casualty numbers
   - missing_or_found_people: Mentions people who are missing, found, or rescued by name or count
   - not_humanitarian: Irrelevant content, ads, jokes, political messages, misinformation
   - other_relevant_information: Weather data, satellite images, locations without people/damage
   - rescue_volunteering_or_donation_effort: Mentions donations, rescue missions, aid, volunteers
   - vehicle_damage: References damaged cars, trucks, ambulances, buses

STEP 2: Analyze the IMAGE separately and thoroughly:
---------------------------------------------------
1. Image informativeness - evaluate visual evidence:
   - informative: Contains ANY visual evidence of disaster impact/response (even if subtle)
   - not_informative: NO visual evidence of disaster impact (ONLY use if completely unrelated)

2. Image category - identify the DOMINANT visual element (choose only ONE):
   - affected_individuals: People appearing distressed/displaced WITHOUT visible injuries
   - infrastructure_and_utility_damage: Visible damage to buildings, roads, bridges
   - injured_or_dead_people: Visible injuries, bodies, medical care
   - missing_or_found_people: Search/rescue scenes, missing posters
   - not_humanitarian: ONLY use when image shows NO disaster-related content whatsoever
   - other_relevant_information: Maps, satellite images, generic disaster scenes without people/objects
   - rescue_volunteering_or_donation_effort: Aid distribution, rescue operations, emergency responders
   - vehicle_damage: Visibly damaged vehicles of any type

3. Damage severity (objective visual assessment):
   - little_or_no_damage: NO VISIBLE structural damage in the image
   - mild_damage: MINOR visible damage (cracking, partial damage, leaning)
   - severe_damage: MAJOR structural collapse or complete destruction
   - dont_know_or_cant_judge: ONLY use when damage cannot be assessed (no structures visible)

IMPORTANT REMINDERS:
- The tweet and image should be analyzed INDEPENDENTLY - they may belong to different categories
- Only classify as "not_humanitarian" when you are CERTAIN there is no disaster-related content
- Look carefully for subtle visual evidence before determining informativeness
- For damage severity, only consider VISIBLE evidence in the image itself

Return classification according to the specified JSON schema."""
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{image_base64}"
                            }
                        }
                    ]
                }
            ],
            "max_tokens": 4096,
            "temperature": 0,
            "response_format": {
                "type": "json_object",
                "schema": {
                    "type": "object",
                    "properties": {
                        "text_analysis": {
                            "type": "object",
                            "properties": {
                                "informativeness": {
                                    "type": "string",
                                    "enum": ["informative", "not_informative"]
                                },
                                "category": {
                                    "type": "string",
                                    "enum": [
                                        "affected_individuals",
                                        "infrastructure_and_utility_damage",
                                        "injured_or_dead_people",
                                        "missing_or_found_people",
                                        "not_humanitarian",
                                        "other_relevant_information",
                                        "rescue_volunteering_or_donation_effort",
                                        "vehicle_damage"
                                    ]
                                }
                            },
                            "required": ["informativeness", "category"]
                        },
                        "image_analysis": {
                            "type": "object",
                            "properties": {
                                "informativeness": {
                                    "type": "string",
                                    "enum": ["informative", "not_informative"]
                                },
                                "category": {
                                    "type": "string",
                                    "enum": [
                                        "affected_individuals",
                                        "infrastructure_and_utility_damage",
                                        "injured_or_dead_people",
                                        "missing_or_found_people",
                                        "not_humanitarian",
                                        "other_relevant_information",
                                        "rescue_volunteering_or_donation_effort",
                                        "vehicle_damage"
                                    ]
                                },
                                "damage_severity": {
                                    "type": "string",
                                    "enum": ["little_or_no_damage", "mild_damage", "severe_damage", "dont_know_or_cant_judge"]
                                }
                            },
                            "required": ["informativeness", "category", "damage_severity"]
                        }
                    },
                    "required": ["text_analysis", "image_analysis"]
                }
            }
        }

        for attempt in range(3):
            try:
                response = requests.post(server_endpoint, json=request_data, timeout=90)
                if response.status_code == 200:
                    result = response.json()
                    if "choices" in result and len(result["choices"]) > 0:
                        content = result["choices"][0]["message"]["content"]
                        try:
                            return json.loads(content)
                        except json.JSONDecodeError:
                            if "```json" in content:
                                json_match = re.search(r'```json\s*([\s\S]*?)\s*```', content)
                                if json_match:
                                    return json.loads(json_match.group(1).strip())
                    if isinstance(result, dict) and "text_analysis" in result:
                        return result
                    if attempt < 2:
                        time.sleep((2 ** attempt) + random.uniform(0, 1))
                        continue
                    return None
                else:
                    if attempt < 2:
                        time.sleep((2 ** attempt) + random.uniform(0, 1))
                    else:
                        return None
            except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as e:
                if attempt < 2:
                    time.sleep((2 ** attempt) + random.uniform(0, 1))
                else:
                    return None
            except Exception as e:
                traceback.print_exc()
                return None
        return None
    except Exception as e:
        traceback.print_exc()
        return None

# --- Config ---
BASE_DIR = '/content/llama.cpp/'
N_SAMPLES = 500
RANDOM_SEED = 42

PRED_COLS = [
    'predicted_text_info',
    'predicted_text_human',
    'predicted_image_info',
    'predicted_image_human',
    'predicted_image_damage',
]

# --- Load original results ---
print(f'\nLoading original results from {RESULTS_FILE}...')
orig_df = pd.read_csv(RESULTS_FILE)
print(f'Original results: {len(orig_df)} samples')

# --- Sample 500 ---
sample_df = orig_df.sample(n=N_SAMPLES, random_state=RANDOM_SEED).copy().reset_index(drop=True)
print(f'Sampled {len(sample_df)} items for retest (seed={RANDOM_SEED})')

# --- Re-classify ---
mismatches = []
match_count = 0
error_count = 0

print(f'\nRe-classifying {N_SAMPLES} samples...\n')
start_time = time.time()

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc='Retest'):
    try:
        # Build image path - try all possible locations
        image_path_in_tsv = row['image_path']
        image_path = None
        candidates = [
            os.path.join('/content/llama.cpp/', image_path_in_tsv),
            os.path.join('/content/llama.cpp/CrisisMMD_v2.0/', image_path_in_tsv),
            os.path.join('/content/', image_path_in_tsv),
            os.path.join('/content/llama.cpp/CrisisMMD_v2.0/', image_path_in_tsv.split('CrisisMMD_v2.0/')[-1] if 'CrisisMMD_v2.0/' in image_path_in_tsv else image_path_in_tsv),
        ]
        for c in candidates:
            if os.path.exists(c):
                image_path = c
                break

        if image_path is None:
            if error_count == 0:  # Print FIRST failure only
                print(f"PATH DEBUG - csv value: '{image_path_in_tsv}'")
                for c in candidates:
                    print(f"  tried: {c}")
            error_count += 1
            continue

        result = classify_with_schema(row['tweet_text'], image_path)

        if result is None:
            error_count += 1
            continue

        new_preds = {
            'predicted_text_info': result['text_analysis']['informativeness'],
            'predicted_text_human': result['text_analysis']['category'],
            'predicted_image_info': result['image_analysis']['informativeness'],
            'predicted_image_human': result['image_analysis']['category'],
            'predicted_image_damage': result['image_analysis']['damage_severity'],
        }

        row_matches = True
        row_diffs = {}
        for col in PRED_COLS:
            orig_val = str(row[col]).strip()
            new_val = str(new_preds[col]).strip()
            if orig_val != new_val:
                row_matches = False
                row_diffs[col] = {'original': orig_val, 'retest': new_val}

        if row_matches:
            match_count += 1
        else:
            mismatches.append({
                'tweet_id': str(row['tweet_id']),
                'image_id': str(row['image_id']),
                'differences': row_diffs,
            })

    except Exception as e:
        print(f'[ERROR] {row.get("tweet_id", "?")}: {e}')
        error_count += 1

elapsed = time.time() - start_time
tested = match_count + len(mismatches)

# --- Report ---
print('\n' + '=' * 60)
print('DETERMINISM RETEST RESULTS')
print('=' * 60)
print(f'Samples tested:    {tested} / {N_SAMPLES}')
print(f'Errors/skipped:    {error_count}')
print(f'Exact matches:     {match_count} / {tested} ({100*match_count/tested:.1f}%)')
print(f'Mismatches:        {len(mismatches)} / {tested} ({100*len(mismatches)/tested:.1f}%)')
print(f'Runtime:           {elapsed:.0f}s ({elapsed/tested:.1f}s per sample)')
print('=' * 60)

if mismatches:
    print(f'\nMismatch details ({len(mismatches)} samples):')
    for m in mismatches[:20]:
        print(f"\n  Tweet {m['tweet_id']} / Image {m['image_id']}:")
        for col, diff in m['differences'].items():
            print(f"    {col}: '{diff['original']}' -> '{diff['retest']}'")
    if len(mismatches) > 20:
        print(f'\n  ... and {len(mismatches) - 20} more (see JSON report)')
else:
    print('\n PERFECT DETERMINISM: All predictions identical across runs.')

# --- Save report ---
report = {
    'test': 'determinism_retest_500',
    'n_samples': N_SAMPLES,
    'random_seed': RANDOM_SEED,
    'tested': tested,
    'errors': error_count,
    'matches': match_count,
    'mismatches_count': len(mismatches),
    'match_rate': round(match_count / tested, 4) if tested > 0 else None,
    'runtime_seconds': round(elapsed, 1),
    'mismatches': mismatches,
}

report_path = '/content/determinism_retest_500.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f'\nReport saved: {report_path}')

# --- Paper sentence ---
if tested > 0:
    print('\n' + '=' * 60)
    print('SUGGESTED SENTENCE FOR SECTION 3.4:')
    print('=' * 60)
    if len(mismatches) == 0:
        print(f'"A reproducibility verification re-classifying {tested} randomly '
              f'sampled items yielded identical predictions across runs, confirming '
              f'deterministic behavior at temperature=0 on identical hardware."')
    else:
        print(f'"A reproducibility verification re-classifying {tested} randomly '
              f'sampled items yielded a {100*match_count/tested:.1f}% exact match rate '
              f'across runs at temperature=0, with {len(mismatches)} minor discrepancies '
              f'attributable to floating-point non-determinism in GPU computation."')

# --- Download ---
files.download(report_path)

Upload your all_results.csv from the original run:


Saving all_results.csv to all_results (7).csv
Uploaded: all_results (7).csv

Loading original results from /content/all_results (7).csv...
Original results: 18082 samples
Sampled 500 items for retest (seed=42)

Re-classifying 500 samples...



Retest:   0%|          | 0/500 [00:00<?, ?it/s]


DETERMINISM RETEST RESULTS
Samples tested:    500 / 500
Errors/skipped:    0
Exact matches:     449 / 500 (89.8%)
Mismatches:        51 / 500 (10.2%)
Runtime:           2251s (4.5s per sample)

Mismatch details (51 samples):

  Tweet 920613959643705344 / Image 920613959643705344_0:
    predicted_image_human: 'rescue_volunteering_or_donation_effort' -> 'other_relevant_information'

  Tweet 905461810828070913 / Image 905461810828070913_0:
    predicted_image_damage: 'little_or_no_damage' -> 'dont_know_or_cant_judge'

  Tweet 922606315737726977 / Image 922606315737726977_0:
    predicted_text_info: 'not_informative' -> 'informative'

  Tweet 910214913796263936 / Image 910214913796263936_0:
    predicted_image_human: 'rescue_volunteering_or_donation_effort' -> 'affected_individuals'

  Tweet 921754220407263237 / Image 921754220407263237_2:
    predicted_image_info: 'informative' -> 'not_informative'
    predicted_image_human: 'rescue_volunteering_or_donation_effort' -> 'not_humanitarian'


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
# Compute F1 on the 500-sample subset for both original and retest
# Load the report
import json
with open('/content/determinism_retest_500.json') as f:
    report = json.load(f)
print(f"Match rate: {report['match_rate']}")
print(f"Mismatches by field:")
from collections import Counter
field_counts = Counter()
for m in report['mismatches']:
    for field in m['differences']:
        field_counts[field] += 1
for field, count in field_counts.most_common():
    print(f"  {field}: {count}")

Match rate: 0.898
Mismatches by field:
  predicted_image_human: 18
  predicted_text_human: 17
  predicted_text_info: 14
  predicted_image_damage: 12
  predicted_image_info: 11
